# Stage 5: global MIEC model

Fits the measured conductivity surface σ(p(O₂), T) of each process with the three-channel mixed ionic-electronic model in a single global fit (6 parameters: three prefactors σ₀ and three activation energies Eₐ). This is the backward approach: from the data to the physical parameters.

**Reads:** `{sample_id}/Results/{condition}/stage3_fit.xlsx`
**Writes:** `{sample_id}/Results/pO2/stage5_model.xlsx` · `Results/pO2/Stage5_*` figures · `session.json → stage5_params`

## Quick links
- Configuration: `SAMPLE_ID`, `MODEL_PEAK_IDS`, `MODEL_EXPONENT`, `MODEL_CONDITIONS`, `MODEL_T_MIN`, `MODEL_T_MAX`
- Step 1: global fit, figures and export
- Step 2: interactive refit (select conditions / temperatures)

**Prerequisite:** run `stage3_drt.ipynb` first; Stage 5 reads the fitted σ per peak from `stage3_fit.xlsx`. p(O₂) data (stages 0 + 1) are required, otherwise the stage skips cleanly.

**Model:** σ(pO₂,T) = (σ₀_ion/T)·e^(−Eₐ_ion/kT) + (σ₀_p/T)·e^(−Eₐ_p/kT)·pO₂^(+x) + (σ₀_n/T)·e^(−Eₐ_n/kT)·pO₂^(−x). A single parameter set must fit the whole surface; that constancy is the physical-validity test.

In [ ]:
import sys
from pathlib import Path
from pipeline.interactive import select_sample
from pipeline.session import load_sample, update_sample

NOTEBOOK_DIR = Path.cwd()

sample_id = select_sample(NOTEBOOK_DIR, show_list=True)

_cfg = load_sample(sample_id)

def _update_session(**fields):
    update_sample(sample_id, **fields)

# USE_SAVED_PARAMS: True resumes the configuration saved in session.json
# (normal use). False writes the values in THIS cell to session.json:
# edit below, set False, run once, then set back to True.
USE_SAVED_PARAMS = True

_p5 = _cfg.get("stage5_config", {}) if USE_SAVED_PARAMS else {}
print("Stage 5 config source:",
      "session.json (saved)" if _p5 else "notebook values (will overwrite session.json)")

# Processes (Zarc peaks) to fit; [] = every peak found in the data
MODEL_PEAK_IDS = _p5.get("MODEL_PEAK_IDS", [])

# Brouwer exponent x: 1/4 in the dilute defect regime, 1/6 elsewhere
MODEL_EXPONENT = _p5.get("MODEL_EXPONENT", 0.25)

# Conditions (pressures) to include in the fit; [] = all
MODEL_CONDITIONS = _p5.get("MODEL_CONDITIONS", [])

# Temperature window [C] for the fit (None = no limit). The ionic/electronic
# separation is physically reliable only where the peaks separate, e.g.
# MODEL_T_MIN = 475 to drop the lower temperatures where they merge.
MODEL_T_MIN = _p5.get("MODEL_T_MIN", None)
MODEL_T_MAX = _p5.get("MODEL_T_MAX", None)

# Written unconditionally (like stage 3): with True this rewrites the saved
# values unchanged (resume); with False it persists the notebook values above.
_update_session(stage5_config={
    "MODEL_PEAK_IDS": MODEL_PEAK_IDS, "MODEL_EXPONENT": MODEL_EXPONENT,
    "MODEL_CONDITIONS": MODEL_CONDITIONS,
    "MODEL_T_MIN": MODEL_T_MIN, "MODEL_T_MAX": MODEL_T_MAX,
})

In [ ]:
# inline backend: more reliable than ipympl with ipywidgets panels.
get_ipython().run_line_magic("matplotlib", "inline")  # type: ignore[name-defined]

import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from pipeline.model import (
    fit_global_conductivity,
    stoichiometric_pO2,
    global_transference_table,
)
from pipeline.plots import (
    apply_pub_style,
    plot_brouwer_transference,
    plot_conductivity_surface_3d,
    plot_fit_residuals,
)
from pipeline.utils import build_metadata_sheet

apply_pub_style()

try:
    import ipywidgets as W
    from IPython.display import display as _display, clear_output as _clear
    _HAS_WIDGETS = True
except Exception as _exc:
    print(f"[INFO] ipywidgets not installed ({_exc}); control panels disabled.")
    _HAS_WIDGETS = False

sample_dir   = NOTEBOOK_DIR / sample_id
RESULTS_BASE = sample_dir / "Results"
print(f"Sample: {sample_id}")

## Step 1: global fit, figures and export

For each selected peak: aggregates σ(p(O₂),T) across the selected conditions and temperature window, fits the 6 parameters (VARPRO), and draws the Stage-4 Brouwer + transference figure (redrawn from the refined global model), the 3-D surface and the residual map. The activation energies are reported as numbers (printout + Parameters sheet); no model-Arrhenius plot is drawn, since by construction it would be a perfect line and hide the real scatter. A structureless residual map and a high global R² mean the data are described by the model. Results go to `stage5_model.xlsx` (Parameters, Residuals, Metadata) and to `session.json` under `stage5_params` (per peak).

In [ ]:
# Aggregate the Peaks rows from every condition that completed Stage 3.
def _load_all_peaks() -> pd.DataFrame:
    frames = []
    if not RESULTS_BASE.exists():
        return pd.DataFrame()
    for d in sorted(RESULTS_BASE.iterdir()):
        f = d / "stage3_fit.xlsx"
        if d.is_dir() and f.exists():
            try:
                df = pd.read_excel(f, sheet_name="Peaks")
            except Exception as exc:
                print(f"[WARN] could not read {f.name}: {type(exc).__name__}: {exc}")
                continue
            if "condition" not in df.columns:
                df["condition"] = d.name
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


df_all_peaks = _load_all_peaks()

if df_all_peaks.empty:
    print("[SKIP] Stage 5: no stage3_fit.xlsx found. Run Stage 3 for this sample first.")
elif not ("pO2_mean" in df_all_peaks.columns and df_all_peaks["pO2_mean"].notna().any()):
    print("[SKIP] Stage 5: no pO2 data available "
          "(pO2 is recorded only with furnace-log data, stages 0 + 1).")
else:
    if MODEL_CONDITIONS:
        df_all_peaks = df_all_peaks[df_all_peaks["condition"].isin(MODEL_CONDITIONS)]
    peak_ids = (MODEL_PEAK_IDS if MODEL_PEAK_IDS
                else sorted(int(p) for p in df_all_peaks["peak_id"].unique()))
    out_dir = RESULTS_BASE / "pO2"
    param_rows, resid_frames, saved = [], [], {}

    for pid in peak_ids:
        df_peak = df_all_peaks[df_all_peaks["peak_id"] == pid]
        try:
            res = fit_global_conductivity(df_peak, x=MODEL_EXPONENT,
                                          t_min=MODEL_T_MIN, t_max=MODEL_T_MAX)
        except ValueError as exc:
            print(f"[SKIP] peak {pid}: {exc}")
            continue
        p, e = res["params"], res["perr"]
        print(f"Peak {pid}: R2={res['r2']:.4f}  n={res['n_points']}  "
              f"Ea_ion={p.Ea_ion:.3f}±{e['Ea_ion']:.3f}  "
              f"Ea_p={p.Ea_p:.3f}±{e['Ea_p']:.3f}  "
              f"Ea_n={p.Ea_n:.3f}±{e['Ea_n']:.3f} eV")
        # Conductivity minimum (n=p crossover): the pO2 AT WHICH sigma is lowest, per T.
        _po2 = df_peak["pO2_mean"].dropna()
        _lo, _hi = (_po2.min(), _po2.max()) if len(_po2) else (None, None)
        for _Tc in sorted(df_peak["T_nominal"].dropna().unique()):
            _pmin = float(stoichiometric_pO2(p, float(_Tc) + 273.15))
            if not np.isfinite(_pmin):
                continue
            _note = "" if (_lo is not None and _lo <= _pmin <= _hi) else "  (extrapolated)"
            print(f"    sigma minimum @ {int(_Tc)} C: pO2 = {_pmin:.2e} bar{_note}")
        # Stage-4 Brouwer + transference figure, redrawn from the refined global
        # model (no fake Arrhenius: the Ea values are reported as numbers above
        # and in the Parameters sheet).
        gtab = global_transference_table(df_peak, p, exponent=MODEL_EXPONENT)
        plot_brouwer_transference(df_peak, out_dir, sample_name=sample_id, peak_id=pid,
                                  exponent=MODEL_EXPONENT, df_t=gtab)
        plot_conductivity_surface_3d(df_peak, p, out_dir, sample_name=sample_id, peak_id=pid)
        plot_fit_residuals(df_peak, p, out_dir, sample_name=sample_id, peak_id=pid)
        plt.show()
        row = {
            "peak_id": pid, "R2": res["r2"], "n_points": res["n_points"], "x": p.x,
            "sigma0_ion": p.sigma0_ion, "Ea_ion": p.Ea_ion, "Ea_ion_err": e["Ea_ion"],
            "sigma0_p": p.sigma0_p, "Ea_p": p.Ea_p, "Ea_p_err": e["Ea_p"],
            "sigma0_n": p.sigma0_n, "Ea_n": p.Ea_n, "Ea_n_err": e["Ea_n"],
        }
        param_rows.append(row)
        rf = res["residuals"].copy()
        rf.insert(0, "peak_id", pid)
        resid_frames.append(rf)
        saved[str(pid)] = {k: v for k, v in row.items() if k != "peak_id"}

    if param_rows:
        df_params = pd.DataFrame(param_rows)
        df_resid  = pd.concat(resid_frames, ignore_index=True)
        df_meta   = build_metadata_sheet(sample_id, "stage5_model", {
            "MODEL_EXPONENT":   MODEL_EXPONENT,
            "MODEL_CONDITIONS": MODEL_CONDITIONS or "all",
            "MODEL_T_MIN":      MODEL_T_MIN,
            "MODEL_T_MAX":      MODEL_T_MAX,
            "MODEL_PEAK_IDS":   MODEL_PEAK_IDS or "all",
        })
        out_dir.mkdir(parents=True, exist_ok=True)
        xlsx = out_dir / "stage5_model.xlsx"
        try:
            with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
                df_params.to_excel(w, sheet_name="Parameters", index=False)
                df_resid.to_excel(w,  sheet_name="Residuals",  index=False)
                df_meta.to_excel(w,   sheet_name="Metadata",   index=False)
            print(f"\nSaved: {xlsx.relative_to(NOTEBOOK_DIR)}")
        except Exception as exc:
            print(f"[WARN] could not write {xlsx.name}: {type(exc).__name__}: {exc}")
        _update_session(stage5_params=saved)
        print("Stage 5 parameters written to session.json")
    else:
        print("[SKIP] Stage 5: no peak could be fitted with the current selection.")

## Step 2: interactive refit

Pick a peak and restrict the conditions and temperatures, then press **↻ Refit**. Use it to exclude the low temperatures where the ionic/electronic separation is not reliable (e.g. keep T ≥ 475 °C). The selected peak's parameters are updated in `session.json` on each refit.

In [ ]:
# Interactive refit panel: pick a peak, restrict conditions / temperatures,
# press Refit. The 6-parameter fit is light, so this recomputes (unlike the
# Brouwer replot selector of Stage 4). Figures use display(fig)+close: no leak.
if (_HAS_WIDGETS and not df_all_peaks.empty
        and "pO2_mean" in df_all_peaks.columns
        and df_all_peaks["pO2_mean"].notna().any()):
    _pids  = sorted(int(p) for p in df_all_peaks["peak_id"].unique())
    _conds = sorted(df_all_peaks["condition"].unique())
    _temps = sorted(int(t) for t in df_all_peaks["T_nominal"].unique())

    w_peak  = W.Dropdown(options=_pids, value=_pids[0], description="Peak:",
                         layout=W.Layout(width="180px"))
    w_conds = W.SelectMultiple(options=_conds, value=tuple(_conds), description="Cond:",
                               rows=min(6, len(_conds)), layout=W.Layout(width="440px"))
    w_temps = W.SelectMultiple(options=_temps, value=tuple(_temps), description="T [°C]:",
                               rows=min(8, len(_temps)), layout=W.Layout(width="180px"))
    w_go    = W.Button(description="↻ Refit", button_style="primary",
                       layout=W.Layout(width="120px"))
    _out    = W.Output()

    def _on_refit(_btn=None):
        with _out:
            _clear(wait=True)
            sub = df_all_peaks[(df_all_peaks["peak_id"] == w_peak.value)
                               & (df_all_peaks["condition"].isin(w_conds.value))
                               & (df_all_peaks["T_nominal"].isin(w_temps.value))]
            try:
                res = fit_global_conductivity(sub, x=MODEL_EXPONENT)
            except ValueError as exc:
                print(f"Fit: {exc}")
                return
            p = res["params"]
            print(f"Peak {w_peak.value}: R2={res['r2']:.4f}  n={res['n_points']}  "
                  f"Ea_ion={p.Ea_ion:.3f}  Ea_p={p.Ea_p:.3f}  Ea_n={p.Ea_n:.3f} eV")
            gtab = global_transference_table(sub, p, exponent=MODEL_EXPONENT)
            fig_bt = plot_brouwer_transference(sub, RESULTS_BASE / "pO2", sample_name=sample_id,
                                               peak_id=w_peak.value, exponent=MODEL_EXPONENT, df_t=gtab)
            fig_r = plot_fit_residuals(sub, p, RESULTS_BASE / "pO2", sample_name=sample_id,
                                       peak_id=w_peak.value, save=False)
            for fig in (fig_bt, fig_r):
                if fig is not None:
                    plt.show()
                    plt.close(fig)
            _update_session(stage5_params={str(w_peak.value): {
                "R2": res["r2"], "n_points": res["n_points"], "x": p.x,
                "sigma0_ion": p.sigma0_ion, "Ea_ion": p.Ea_ion,
                "sigma0_p": p.sigma0_p, "Ea_p": p.Ea_p,
                "sigma0_n": p.sigma0_n, "Ea_n": p.Ea_n}})

    w_go.on_click(_on_refit)
    _display(W.VBox([W.HBox([w_peak, w_go]), W.HBox([w_conds, w_temps]), _out]))
    _on_refit()
else:
    print("[INFO] interactive refit panel needs ipywidgets and a fitted dataset.")

## Output summary

`Results/pO2/`:
- `stage5_model.xlsx` — Parameters (6 params + R² + n per peak), Residuals, Metadata
- `Brouwer_transference_Peak*` — σ vs p(O₂) + t_ion, redrawn from the global model
- `Stage5_surface3D_Peak*` — fitted σ(p(O₂),T) surface with measured points
- `Stage5_residuals_Peak*` — relative-residual map over (p(O₂), T)

Each figure is saved as PNG (preview) and PDF (publication).